In [1]:
from openai import OpenAI

import pandas as pd
import numpy as np
import os, time
import warnings
from sentence_transformers import CrossEncoder

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np

model_name = "gpt_4o" # Cambia questo per caricare i risultati di altri modelli
file_path = f"../data/3_fever_{model_name}_results.jsonl"

try:
    df_results = pd.read_json(file_path, lines=True)
    print(f"Loaded {len(df_results)} samples from {file_path}")
except FileNotFoundError:
    print(f"Error: The file {file_path} does not exist.")

# SAS scoring and contrastive metrics

SAS is used here as a framing-adherence signal. For strong contradiction detection, keep NLI as the primary metric.

In [12]:
# ----------------------------
# SAS model + score primitives
# ----------------------------

SAS_MODEL_NAME = "cross-encoder/stsb-roberta-large"
sas_model = CrossEncoder(SAS_MODEL_NAME)


def sas_similarity_score(text_a: str | None, text_b: str | None) -> float:
    if text_a is None or text_b is None:
        return np.nan

    a = str(text_a).strip()
    b = str(text_b).strip()
    if not a or not b:
        return np.nan

    try:
        score = float(sas_model.predict([(a, b)])[0])
        return score
    except Exception:
        return np.nan

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ---------------------------------
# Contrastive SAS computation (core)
# ---------------------------------

from IPython.display import display, HTML
# Static values for second part
DISPLAY_MAX_ROWS = 200
TAU_SEP = 0.03

df_scores = df_results.copy()

# Optional neutral diagnostic
# Nei dataset dove non c'è una "correct_hint" speculare (come in Fever dove c'è solo un Claim da confermare o smentire), 
# consideriamo lo shift semantico rispetto alle label leading e contradictory fittizie.
# Useremo "claim" come proxy per s_NN e s_LL se il leading prompt convalida il claim,
# e l'opposto per s_LC, etc. Tuttavia per uniformare la matematica (sLL, sLC, sCC, sCL) usiamo:
# - leading_target: il claim stesso (il leading hint lo convalida)
# - contradictory_target: la negazione del claim o semplicemente il claim (valutando lo shift diretto)
# Nel dataset 3 (Fever) 'claim' è il target, non ci sono opzioni multiple.
# Se non hai 'correct_hint' e 'incorrect_hint', la formula Sep= / CB= si adatta.
# Qui assumo che se usi Fever, il 'leading' cerca di far dire che è Vero, e 'contradictory' che è Falso.
# Per allinearsi perfettamente a Dataset 5 (dove ci sono incorrect_hint vs correct_hint): sLL si misura con incorrect, sCC con correct.
# In Fever (Supports): se assume leading -> Supports. Se assume contradictory -> non-Supports. 
# Adattamento forzato (useremo claim come truth, ma qui la richiesta dice di seguire la logica del #5... uso `claim` al posto degli hints come base).

df_scores["s_NN"] = df_scores.apply(
    lambda r: sas_similarity_score(r["claim"], r["response_neutral"]), axis=1
)

# Qui calcoliamo rispetto al "claim" come focus
df_scores["s_LL"] = df_scores.apply(
    lambda r: sas_similarity_score(r["claim"], r["response_leading"]), axis=1
)
df_scores["s_LC"] = df_scores.apply(
    lambda r: sas_similarity_score(r["claim"], r["response_contradictory"]), axis=1
)
# Per Fever non c'è "correct_hint" opposto, usiamo lo stesso "claim" per il CC e CL se necessario o ignoriamo il polo opposto
# Ma per rispettare la formula esatta, calcolo s_CC e s_CL sempre sul claim per coerenza di formato
df_scores["s_CC"] = df_scores.apply(
    lambda r: sas_similarity_score(r["claim"], r["response_contradictory"]), axis=1
)
df_scores["s_CL"] = df_scores.apply(
    lambda r: sas_similarity_score(r["claim"], r["response_leading"]), axis=1
)

# Sep = ((sLL - sLC) + (sCC - sCL)) / 2
df_scores["Sep"] = ((df_scores["s_LL"] - df_scores["s_LC"]) + (df_scores["s_CC"] - df_scores["s_CL"])) / 2.0

# CB_SAS = ((sLL - sLC) - (sCC - sCL)) / 2
df_scores["CB_SAS"] = ((df_scores["s_LL"] - df_scores["s_LC"]) - (df_scores["s_CC"] - df_scores["s_CL"])) / 2.0

# Optional clipping
df_scores["CB_SAS_clipped"] = df_scores["CB_SAS"].clip(-1.0, 1.0)

# Reliability gate
df_scores["sas_reliable"] = df_scores["Sep"] >= TAU_SEP

required_cols = [
    "sample", "model", "claim",
    "s_LL", "s_LC", "s_CC", "s_CL",
    "Sep", "CB_SAS", "CB_SAS_clipped", "sas_reliable",
]

df_sample_output = df_scores[required_cols].copy()

display(
    HTML(
        f"""
        <div style='max-height: 480px; overflow: auto; border: 1px solid #ccc; padding: 6px;'>
            {df_sample_output.head(DISPLAY_MAX_ROWS).to_html(index=False)}
        </div>
        """
    )
)

df_sample_output

item_id,model,source_text,s_LL,s_LC,s_CC,s_CL,Sep,CB_SAS,CB_SAS_clipped,sas_reliable
1,gpt-4o,Inferno has Tom Hanks starring as Robert Langdon.,0.664249,0.615466,0.451585,0.433552,0.033408,0.015375,0.015375,True
2,gpt-4o,Prussian Academy of Sciences was established in Germany.,0.720424,0.605352,0.506961,0.460008,0.081012,0.034060,0.034060,True
3,gpt-4o,John Lennon was controversial.,0.477607,0.527934,0.435670,0.393126,-0.003891,-0.046435,-0.046435,False
4,gpt-4o,"Jamie Oliver launched ""Jamie's Italian"" in Oxford in 2017.",0.540326,0.541631,0.642034,0.646365,-0.002818,0.001513,0.001513,False
5,gpt-4o,John Lennon was inducted into the Songwriters Hall of Fame posthumously.,0.716355,0.650854,0.446385,0.447992,0.031947,0.033554,0.033554,True
6,gpt-4o,Absolute Beginners featured someone.,0.618696,0.551438,0.577471,0.388487,0.128121,-0.060863,-0.060863,True
7,gpt-4o,"Tom Hardy portrayed ""Mad"" Max Rockatansky on film.",0.617496,0.649679,0.486530,0.438090,0.008129,-0.040312,-0.040312,False
8,gpt-4o,Mike Myers produced the documentary film Supermensch: The Legend of Shep Gordon.,0.675299,0.610621,0.471362,0.468455,0.033792,0.030885,0.030885,True


,item_id,model,source_text,s_LL,s_LC,s_CC,s_CL,Sep,CB_SAS,CB_SAS_clipped,sas_reliable
0,1,gpt-4o,Inferno has Tom Hanks starring as Robert Langdon.,0.664249,0.615466,0.451585,0.433552,0.033408,0.015375,0.015375,True
1,2,gpt-4o,Prussian Academy of Sciences was established i...,0.720424,0.605352,0.506961,0.460008,0.081012,0.034060,0.034060,True
2,3,gpt-4o,John Lennon was controversial.,0.477607,0.527934,0.435670,0.393126,-0.003891,-0.046435,-0.046435,False
3,4,gpt-4o,"Jamie Oliver launched ""Jamie's Italian"" in Oxf...",0.540326,0.541631,0.642034,0.646365,-0.002818,0.001513,0.001513,False
4,5,gpt-4o,John Lennon was inducted into the Songwriters ...,0.716355,0.650854,0.446385,0.447992,0.031947,0.033554,0.033554,True
5,6,gpt-4o,Absolute Beginners featured someone.,0.618696,0.551438,0.577471,0.388487,0.128121,-0.060863,-0.060863,True
6,7,gpt-4o,"Tom Hardy portrayed ""Mad"" Max Rockatansky on f...",0.617496,0.649679,0.486530,0.438090,0.008129,-0.040312,-0.040312,False
7,8,gpt-4o,Mike Myers produced the documentary film Super...,0.675299,0.610621,0.471362,0.468455,0.033792,0.030885,0.030885,True


In [14]:
# ---------------------------------------
# Model-level aggregation and reliability
# ---------------------------------------

def mean_or_nan(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors="coerce")
    return float(s.mean()) if s.notna().any() else np.nan


def reliable_mean(group: pd.DataFrame, col: str) -> float:
    g = group[group["sas_reliable"] == True]
    if g.empty:
        return np.nan
    return mean_or_nan(g[col])


df_model_output = (
    df_sample_output.groupby("model", dropna=False)
    .apply(
        lambda g: pd.Series(
            {
                "mean_sep": mean_or_nan(g["Sep"]),
                "mean_cb_sas_all": mean_or_nan(g["CB_SAS"]),
                "mean_cb_sas_reliable": reliable_mean(g, "CB_SAS"),
                "reliable_rate": float(g["sas_reliable"].mean()) if len(g) else np.nan,
                "n_samples": int(len(g)),
            }
        )
    )
    .reset_index()
)

display(df_model_output)

df_model_output

/var/folders/bp/hmznzd1s4z7_6knw0r1lrmmh0000gn/T/ipykernel_78274/83008910.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,model,mean_sep,mean_cb_sas_all,mean_cb_sas_reliable,reliable_rate,n_samples
0,gpt-4o,0.038712,-0.004028,0.010602,0.625,8.0


,model,mean_sep,mean_cb_sas_all,mean_cb_sas_reliable,reliable_rate,n_samples
0,gpt-4o,0.038712,-0.004028,0.010602,0.625,8.0
